# OpenAI Unstructured-Output Evaluation

Evaluate a selected OpenAI model on Calculus I multiple-choice questions. The primary model produces a free-form solution, and a smaller extraction model converts that solution into a structured answer choice for evaluation.

## 1. Imports and configuration

In [1]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

### Run configuration

Change `MODEL_NAME` to test another primary model. The artifact directory and every generated filename update automatically.

In [2]:
# Support launching Jupyter from either the repository root or notebooks/.
WORKING_DIRECTORY = Path.cwd()
PROJECT_ROOT = (
    WORKING_DIRECTORY.parent
    if WORKING_DIRECTORY.name == "notebooks"
    else WORKING_DIRECTORY
)

MODEL_NAME = "gpt-4o-2024-08-06"
EXTRACTION_MODEL_NAME = "gpt-4o-mini"
RUN_NAME = f"{MODEL_NAME}-subsample"

DATA_PATH = PROJECT_ROOT / "data" / "test3_input_subset_seed42_numq1 (clean testing dataset).csv"
MODEL_OUTPUT_DIRECTORY = PROJECT_ROOT / "output" / MODEL_NAME
CHECKPOINT_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-checkpoint.json"
RESULTS_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output.csv"
INCORRECT_RESULTS_PATH = (
    MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output-incorrect.csv"
)

MODEL_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

## 2. OpenAI client

In [3]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 3. Load and inspect the dataset

In [4]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the evaluation dataset."""
    return pd.read_csv(data_path)


def summarize_dataset(dataframe: pd.DataFrame) -> pd.Series:
    """Summarize row counts and duplicate checks used in the original notebook."""
    return pd.Series(
        {
            "rows": len(dataframe),
            "columns": len(dataframe.columns),
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "duplicate_questions": int(dataframe["Question"].duplicated().sum()),
            "duplicate_ids": int(dataframe["ID"].duplicated().sum()),
        }
    )


df = load_dataset(DATA_PATH)
print(f"Loaded {len(df)} questions from {DATA_PATH}")
summarize_dataset(df)

Loaded 331 questions from /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/data/test3_input_subset_seed42_numq1 (clean testing dataset).csv


rows                   331
columns                  3
duplicate_rows           0
duplicate_questions      0
duplicate_ids            0
dtype: int64

In [5]:
# Inspect any duplicated rows, questions, or IDs found by the summary above.
duplicate_rows = df.loc[df.duplicated(keep=False)].sort_values(df.columns.tolist())
duplicate_questions = df.loc[df["Question"].duplicated(keep=False)].sort_values(
    "Question"
)
duplicate_ids = df.loc[df["ID"].duplicated(keep=False)].sort_values("ID")

df.describe(include="all")

,ID,Question,Correct Answer
count,331,331,331
unique,331,331,7
top,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b
freq,1,1,169


## 4. Unstructured response pipeline

### Few-shot example

In [6]:
EXAMPLE_RESPONSE = '''
"Complete the square to write the equation in the standard form f(x)=a(x-h)^2+k. f(x)=2x^2+12x+25
                a) f(x)=2(x+3)^2+16
                b) f(x)=2(x-6)^2+7
                c) f(x)=2(x+3)^2+7
                d) f(x)=2(x-3)^2+16
                e) f(x)=2(x+3)^2-7
                f) None of the above."
    "detailed_solution_explanation": "To complete the square for f(x) = 2x² + 12x + 25, first factor out 2 from the terms involving x:\n\nf(x) = 2(x² + 6x) + 25.\n\nNext, complete the square inside the parentheses:\n\nx² + 6x = x² + 6x + 9 − 9 = (x + 3)² − 9.\n\nRewrite f(x):\n\nf(x) = 2[(x + 3)² − 9] + 25\n      = 2(x + 3)² − 18 + 25\n      = 2(x + 3)² + 7.\n",
    "correct_option_choice_letter": "c"
'''

SYSTEM_PROMPT = (
    "Act as a mathematician specializing in Calculus. Solve the college-level "
    "Calculus I multiple-choice problem step by step and provide the correct "
    "option letter."
)

In [7]:
def load_checkpoint(checkpoint_path: Path) -> dict:
    """Load saved primary-model responses, or return an empty checkpoint."""
    if not checkpoint_path.exists():
        return {}
    with checkpoint_path.open("r", encoding="utf-8") as checkpoint_file:
        return json.load(checkpoint_file)


def save_checkpoint(checkpoint_path: Path, data: dict) -> None:
    """Persist primary-model responses as formatted JSON."""
    with checkpoint_path.open("w", encoding="utf-8") as checkpoint_file:
        json.dump(data, checkpoint_file, indent=4)

In [8]:
def query_openai(question: str, model: str = MODEL_NAME) -> str:
    """Request one free-form calculus solution from the primary model."""
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Use this example as a reference:\n{EXAMPLE_RESPONSE}\n\n"
                    f"Now solve the following question:\n{question}"
                ),
            },
        ],
    )
    return completion.choices[0].message.content or ""

In [9]:
def collect_responses(
    dataframe: pd.DataFrame,
    checkpoint_path: Path,
    model: str = MODEL_NAME,
) -> dict:
    """Query unanswered rows and checkpoint each free-form response."""
    checkpoint = load_checkpoint(checkpoint_path)

    for _, row in dataframe.iterrows():
        question_id = str(row["ID"])
        if question_id in checkpoint:
            print(f"Skipping ID {question_id}, already processed.")
            continue

        print(f"Processing ID {question_id}...")
        checkpoint[question_id] = {
            "Long Answer": query_openai(row["Question"], model=model)
        }
        save_checkpoint(checkpoint_path, checkpoint)

    save_checkpoint(checkpoint_path, checkpoint)
    print(f"Checkpoint saved to {checkpoint_path}")
    return checkpoint


def add_explanations(dataframe: pd.DataFrame, checkpoint: dict) -> pd.DataFrame:
    """Add checkpointed free-form solutions to the dataset."""
    result = dataframe.copy()
    result["LLM Answer Explanation"] = result["ID"].astype(str).map(
        lambda question_id: checkpoint.get(question_id, {}).get("Long Answer", "")
    )
    return result

### Run the primary-model pipeline

> **Note:** This cell makes billable primary-model API calls only for questions absent from the checkpoint.

In [10]:
df = df.head(10)  # Limit to first 10 rows for testing

checkpoint = collect_responses(df, CHECKPOINT_PATH)
df = add_explanations(df, checkpoint)
df.head(10)

Processing ID q2_sp22_problem10__04...
Processing ID q27_sp23_problem04__04...
Processing ID q23_sp23_problem09__02...
Processing ID q31_sp23_problem03__09...
Processing ID q10_sp22_problem07__21...
Processing ID q19_fa22_problem03__21...
Processing ID q28_sp23_problem03__04...
Processing ID q25_sp23_problem08__02...
Processing ID q2_sp22_problem06__04...
Processing ID q5_sp22_problem07__01...
Checkpoint saved to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/output/gpt-4o-2024-08-06/gpt-4o-2024-08-06-subsample-checkpoint.json


,ID,Question,Correct Answer,LLM Answer Explanation
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b,To complete the square for the function \( f(x...
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b,"To solve this problem, we will set up the func..."
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b,To differentiate the function \( f(x) = \tan(5...
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d,"To solve this problem, we need to differentiat..."
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c,To find the slope of the tangent line to the g...
5,q19_fa22_problem03__21,21) Find the intervals on which \(f(x)=\displa...,c,To determine the intervals on which the functi...
6,q28_sp23_problem03__04,4) Use differentials to estimate the value of ...,e,We need to use differentials to estimate the v...
7,q25_sp23_problem08__02,2) Differentiate the following: \(f(x)=\displ...,b,To find the derivative of the function \( f(x)...
8,q2_sp22_problem06__04,4) Find the <i>x</i>- and <i>y</i>-intercepts ...,b,To find the x- and y-intercepts for the quadra...
9,q5_sp22_problem07__01,1) Evaluate \(\displaystyle \lim _{x\rightarro...,c,"To evaluate \(\lim_{x \to 0} f(x)\), where \(f..."


## 5. Extract structured answer choices

In [11]:
class AnswerChoice(BaseModel):
    """Structured answer extracted from a free-form solution."""

    correct_option_choice_letter: str


def get_correct_option(
    response_text: str, model: str = EXTRACTION_MODEL_NAME
) -> str:
    """Extract an option letter and return the raw structured JSON text."""
    prompt = (
        "Read the following raw response from an LLM and extract the correct "
        f"option letter:\n\n{response_text}\n\n"
        "If you cannot find the correct option letter, output 'NA'."
    )
    parse_completion = getattr(client.chat.completions, "parse", None)
    if parse_completion is None:
        parse_completion = client.beta.chat.completions.parse

    completion = parse_completion(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert annotator. You convert raw text to "
                    "structured text."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        response_format=AnswerChoice,
    )
    return completion.choices[0].message.content or ""


def extract_answer_choices(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Extract a structured answer choice from each free-form solution."""
    result = dataframe.copy()
    for index, row in result.iterrows():
        explanation = row["LLM Answer Explanation"]
        if pd.isna(explanation) or not explanation:
            continue
        try:
            result.at[index, "LLM Answer Choice (RAW)"] = get_correct_option(
                explanation
            )
        except Exception as error:
            print(f"Error processing ID {row['ID']}: {error}")
            result.at[index, "LLM Answer Choice (RAW)"] = None
    return result

### Run answer extraction

> **Note:** This cell makes one billable extraction-model API call per non-empty solution.

In [12]:
df = extract_answer_choices(df)
df.head()

,ID,Question,Correct Answer,LLM Answer Explanation,LLM Answer Choice (RAW)
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b,To complete the square for the function \( f(x...,"{""correct_option_choice_letter"":""b""}"
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b,"To solve this problem, we will set up the func...","{""correct_option_choice_letter"":""b""}"
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b,To differentiate the function \( f(x) = \tan(5...,"{""correct_option_choice_letter"":""b""}"
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d,"To solve this problem, we need to differentiat...","{""correct_option_choice_letter"":""d""}"
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c,To find the slope of the tangent line to the g...,"{""correct_option_choice_letter"":""c""}"


## 6. Normalize and evaluate answers

In [13]:
def extract_letter_choice(raw_response: str) -> str:
    """Read the option letter from the extraction model's JSON response."""
    if not isinstance(raw_response, str) or not raw_response:
        return ""
    try:
        return json.loads(raw_response).get("correct_option_choice_letter", "")
    except json.JSONDecodeError:
        match = re.search(
            r'"correct_option_choice_letter"\s*:\s*"(.*?)"',
            raw_response,
        )
        return match.group(1) if match else ""


def evaluate_answers(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    """Normalize extracted choices and calculate accuracy."""
    result = dataframe.copy()
    result["LLM Answer Choice"] = result["LLM Answer Choice (RAW)"].map(
        extract_letter_choice
    )
    result["Correct"] = (
        result["Correct Answer"] == result["LLM Answer Choice"]
    ).astype(int)
    return result, result["Correct"].mean()

In [14]:
df, accuracy = evaluate_answers(df)
print(f"Accuracy: {accuracy * 100:.2f}%")
df.head()

Accuracy: 100.00%


,ID,Question,Correct Answer,LLM Answer Explanation,LLM Answer Choice (RAW),LLM Answer Choice,Correct
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b,To complete the square for the function \( f(x...,"{""correct_option_choice_letter"":""b""}",b,1
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b,"To solve this problem, we will set up the func...","{""correct_option_choice_letter"":""b""}",b,1
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b,To differentiate the function \( f(x) = \tan(5...,"{""correct_option_choice_letter"":""b""}",b,1
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d,"To solve this problem, we need to differentiat...","{""correct_option_choice_letter"":""d""}",d,1
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c,To find the slope of the tangent line to the g...,"{""correct_option_choice_letter"":""c""}",c,1


## 7. Export results

In [15]:
incorrect_df = df.loc[df["Correct"] == 0].copy()

incorrect_df.to_csv(INCORRECT_RESULTS_PATH, index=False)
df.to_csv(RESULTS_PATH, index=False)

print(f"Saved {len(df)} results to {RESULTS_PATH}")
print(f"Saved {len(incorrect_df)} incorrect results to {INCORRECT_RESULTS_PATH}")

Saved 10 results to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/output/gpt-4o-2024-08-06/gpt-4o-2024-08-06-subsample-test-output.csv
Saved 0 incorrect results to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/output/gpt-4o-2024-08-06/gpt-4o-2024-08-06-subsample-test-output-incorrect.csv
